In [8]:
import geopandas as gpd
from sqlalchemy import create_engine
 
user = "njteh_select"
password = "\\0#IHiT3,s2m"
host = "wirelesspostgresqlflexible.postgres.database.azure.com"
port = "5432"
database = "wiroidb2"
 
connection_string = f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{database}"
 
engine = create_engine(connection_string)

query = "SELECT * FROM us_road_2025 WHERE state_name='NY';"
 
gdf = gpd.GeoDataFrame.from_postgis(
    query,
    engine,
    geom_col="geom"
)
 
gdf.head()

,id,linear_id,full_name,rttyp,mtfcc,county_fips,geom,upload_date,state_name
0,9123348,110471130926,NaN,NaN,S1400,36003,"MULTILINESTRING ((1366816.386 382885.584, 1366...",2026-01-08 00:24:03.601615,NY
1,9113803,110789254950,Nolan Rd Exd,M,S1400,36001,"MULTILINESTRING ((1698305.814 491147.308, 1698...",2026-01-08 00:24:01.531642,NY
2,9113863,1103357512193,Kirkner Spr,M,S1400,36001,"MULTILINESTRING ((1699814.745 525239.129, 1699...",2026-01-08 00:24:01.531642,NY
3,9114372,110789245223,Hall Pl,M,S1400,36001,"MULTILINESTRING ((1698960.942 514101.537, 1698...",2026-01-08 00:24:01.531642,NY
4,9114575,110789274420,Thistle Ln,M,S1400,36001,"MULTILINESTRING ((1696196.823 507603.685, 1696...",2026-01-08 00:24:01.531642,NY


In [6]:
gdf.to_parquet("edges_NY.parquet", index=False)

  Using cached psycopg2-2.9.11-cp313-cp313-win_amd64.whl.metadata (5.1 kB)
Using cached psycopg2-2.9.11-cp313-cp313-win_amd64.whl (2.7 MB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
from src.data.graph_builder import GraphBuilder
g_g = GraphBuilder()
G = g_g.build_graph(roads = gdf)

In [14]:
g_g = GraphBuilder()
G = g_g.build_graph(roads = gdf)

KeyboardInterrupt: 

In [ ]:
# assume `roads_gdf` is huge, and `locations` is your points GeoDataFrame
bbox = locations.total_bounds  # [minx, miny, maxx, maxy]
buffer_m = 1000  # how far out to keep roads
# convert buffer into degrees approx (roughly 1km ~= 0.009 degrees)
buffer_deg = buffer_m / 111000.0

minx, miny, maxx, maxy = bbox
roads_sub = roads_gdf.cx[minx-buffer_deg : maxx+buffer_deg, miny-buffer_deg : maxy+buffer_deg]

In [16]:
pip install osmnx==1.3.0

  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached charset_normalizer-3.4.5-cp313-cp313-win_amd64.whl.metadata (39 kB)
  Using cached idna-3.11-py3-none-any.whl.metadata (8.4 kB)
  Using cached urllib3-2.6.3-py3-none-any.whl.metadata (6.9 kB)
   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
   -- ------------------------------------- 0.5/8.1 MB 5.1 MB/s eta 0:00:02
   ------ --------------------------------- 1.3/8.1 MB 3.7 MB/s eta 0:00:02
   ------------ --------------------------- 2.6/8.1 MB 4.4 MB/s eta 0:00:02
   ------------------ --------------------- 3.7/8.1 MB 4.8 MB/s eta 0:00:01
   ------------------------ --------------- 5.0/8.1 MB 4.9 MB/s eta 0:00:01
   ----------------------------- ---------- 6.0/8.1 MB 5.0 MB/s eta 0:00:01
   ---------------------------------- ----- 7.1/8.1 MB 5.0 MB/s eta 0:00:01
   ---------------------------------------  8.1/8.1 MB 4.9 MB/s eta 0:00:01
   ---------------------------------------- 8.1/8.


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [17]:
from osmnx.utils_graph import graph_from_gdfs
import pandas as pd
import networkx as nx

def create_graph(road_data):
        
        road_data["start_point"] = road_data.geometry.apply(lambda x: x.coords[0])

        road_data["end_point"] = road_data.geometry.apply(lambda x: x.coords[-1])

        gdf_nodes = pd.DataFrame({"data" : list(set(list(set(road_data["start_point"]))+ list(set(road_data["end_point"]))))})

        gdf_nodes["x"] = gdf_nodes["data"].apply(lambda x : x[0])

        gdf_nodes["y"] = gdf_nodes["data"].apply(lambda x : x[1])

        gdf_nodes["osmid"]= gdf_nodes.index

        dictt= gdf_nodes.set_index("data")["osmid"].to_dict()

        road_data["u"] = road_data["start_point"].map(dictt)

        road_data["v"] = road_data["end_point"].map(dictt)

        road_data["key"] = 0

        gdf_edges = road_data[["u","v","key","geometry"]]

        gdf_edges.set_index(['u', 'v', 'key'], inplace=True)

        gdf_edges["length"] = gdf_edges.geometry.length

        G = graph_from_gdfs(gdf_nodes,gdf_edges)

        G = nx.Graph(G)
        
        return G,gdf_nodes,gdf_edges


In [21]:
gdf = gdf.explode()

In [23]:
gdf = gdf.reset_index(drop=True)

In [28]:
gdf = gdf.rename(columns = {"geom" : "geometry"})
gdf.set_geometry("geometry", inplace=True)

In [29]:
G, gdf_nodes, gdf_edges = create_graph(road_data = gdf)

In [46]:
# inspect dtypes
print(gdf_nodes.dtypes)

# convert any pandas Period columns to string
for col, dt in gdf_nodes.dtypes.items():
    if str(dt).startswith("period"):
        gdf_nodes[col] = gdf_nodes[col].astype(str)

data      object
x        float64
y        float64
osmid      int64
dtype: object


In [57]:
from pandas.api.types import is_period_dtype

# Select only the columns we want to export
# Convert any object/period-like columns to safe types for Parquet

d = pd.DataFrame({
    # "data": gdf_nodes["da114ta"],
    "x": gdf_nodes["x"],
    "y": gdf_nodes["y"],
    "osmid": gdf_nodes["osmid"],
})

print("Export dtypes before conversion:")
print(d.dtypes)

for col in d.columns:
    dt = d[col].dtype
    if is_period_dtype(dt):
        d[col] = d[col].astype(str)
    elif dt == object:
        # If objects contain pandas Period values, convert to strings
        sample = d[col].dropna().head(20)
        if any(isinstance(v, pd.Period) for v in sample):
            d[col] = d[col].astype(str)

print("Export dtypes after conversion:")
print(d.dtypes)

# Try pyarrow first, fall back to fastparquet if needed
try:
    d.to_parquet("nodes_NY.parquet", index=False, engine="pyarrow")
    print("Exported nodes_NY.parquet with pyarrow")
except Exception as e:
    print("pyarrow failed, falling back to fastparquet:", e)
    try:
        d.to_parquet("nodes_NY.parquet", index=False, engine="fastparquet")
        print("Exported nodes_NY.parquet with fastparquet")
    except Exception as e2:
        print("fastparquet also failed:", e2)
        raise

Export dtypes before conversion:
x        float64
y        float64
osmid      int64
dtype: object
Export dtypes after conversion:
x        float64
y        float64
osmid      int64
dtype: object
pyarrow failed, falling back to fastparquet: A type extension with name pandas.period already defined
Exported nodes_NY.parquet with fastparquet


C:\Users\levon\AppData\Local\Temp\ipykernel_11700\4015200693.py:18: Pandas4Warning: is_period_dtype is deprecated and will be removed in a future version. Use `isinstance(dtype, pd.PeriodDtype)` instead
  if is_period_dtype(dt):


In [61]:
import pandas as pd

# nodes_df = pd.read_parquet("nodes_NY.parquet", engine="pyarrow")
edges_df = pd.read_parquet("edges_NY.parquet", engine="pyarrow")

ArrowKeyError: A type extension with name pandas.period already defined

In [1]:
import pandas as pd

nodes_gdf = pd.read_parquet("nodes_NY.parquet", engine="pyarrow")
edges_gdf = pd.read_parquet("edges_NY.parquet", engine="pyarrow")

In [2]:
edges_gdf

,geometry,length
0,"b""\x01\x02\x00\x00\x00\x02\x00\x00\x00\xc04\xc...",66.221718
1,b'\x01\x02\x00\x00\x00\x02\x00\x00\x00\rqh\xd0...,108.204591
2,b'\x01\x02\x00\x00\x00\x02\x00\x00\x00\xbb\x14...,105.105664
3,"b""\x01\x02\x00\x00\x00\x02\x00\x00\x00\xa6\xd0...",95.558834
4,b'\x01\x02\x00\x00\x00\x02\x00\x00\x00Dw\xc5\x...,84.281971
...,...,...
379642,b'\x01\x02\x00\x00\x00\x05\x00\x00\x00\xfa\xa1...,44.842354
379643,b'\x01\x02\x00\x00\x00\x0c\x00\x00\x00\x9d\x03...,307.405761
379644,b'\x01\x02\x00\x00\x00\x0c\x00\x00\x00qU\x89z\...,1304.223640
379645,b'\x01\x02\x00\x00\x00\n\x00\x00\x00\xf7&\x01\...,94.472851


In [3]:
nodes_gdf

,x,y,osmid
0,1.502231e+06,694323.207105,0
1,1.401727e+06,479075.219347,1
2,1.392668e+06,519491.843113,2
3,1.733058e+06,448797.503498,3
4,1.703538e+06,404855.141627,4
...,...,...,...
581293,1.786154e+06,297865.951500,581293
581294,1.468319e+06,558365.784899,581294
581295,1.742737e+06,269497.715583,581295
581296,1.310384e+06,466841.046718,581296


In [16]:
import geopandas as gpd
import shapely 
edges_gdf['geometry'] = edges_gdf['geometry'].apply(lambda x: shapely.from_wkb(x))

In [ ]:
gpd.GeoDataFrame(edges_gdf).to_file("edges_NY.geojson", driver="GeoJSON")